# PPO hyperparameter tuning (Optuna + Portfolio env)

Each **trial**: train PPO on `training_data`, then one validation rollout with `ppo.predict`.  
**Objective**: maximize validation `total_return` from the env `info` dict.

**Kernel**: select the project interpreter `.venv` so imports match your environment.

In [2]:
import os
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import optuna
import pandas as pd
import torch

# Repo root on sys.path (works if notebook cwd is repo root or scripts/)
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "scripts" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from agent.ppo import PPO
from envs.portfolio_env import PortfolioEnvWithBaselines
from src.data_loader import process_data_with_indicators, split_data

In [3]:
DEFAULT_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
    "META", "TSLA", "NFLX", "UNH", "JNJ",
    "V", "JPM", "WMT", "MA", "PG",
    "HD", "DIS", "BAC", "XOM", "CVX",
]

DATA_DIR = REPO_ROOT / "data"


def load_stock_data(tickers: list, data_dir: Path) -> dict:
    stock_data = {}
    for ticker in tickers:
        path = data_dir / f"{ticker}.csv"
        df = pd.read_csv(path, skiprows=2, index_col="Date", parse_dates=True)
        df.columns = ["Open", "High", "Low", "Close", "Volume"]
        stock_data[ticker] = df
    return process_data_with_indicators(stock_data)


def rollout_eval(ppo: PPO, env: PortfolioEnvWithBaselines) -> dict:
    state, _ = env.reset()
    done = False
    info = {}
    while not done:
        action = ppo.predict(state)
        state, _, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    return info


def build_objective(
    training_data: dict,
    validation_data: dict,
    train_iterations: int,
    steps_per_iter: int,
    torch_seed: Optional[int],
):
    def objective(trial: optuna.Trial) -> float:
        if torch_seed is not None:
            s = torch_seed + trial.number
            np.random.seed(s)
            torch.manual_seed(s)

        lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
        gamma = trial.suggest_float("gamma", 0.95, 0.999)
        gae_lambda = trial.suggest_float("gae_lambda", 0.90, 0.99)
        clip_range = trial.suggest_float("clip_range", 0.1, 0.3)
        n_epochs = trial.suggest_int("n_epochs", 4, 15)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
        ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.05, log=True)
        vf_coef = trial.suggest_float("vf_coef", 0.1, 1.0)

        train_env = PortfolioEnvWithBaselines(training_data)
        ppo = PPO(
            env=train_env,
            lr=lr,
            gamma=gamma,
            gae_lambda=gae_lambda,
            clip_range=clip_range,
            n_epochs=n_epochs,
            batch_size=batch_size,
            ent_coef=ent_coef,
            vf_coef=vf_coef,
        )
        ppo.train(n_iterations=train_iterations, steps_per_iter=steps_per_iter)

        val_env = PortfolioEnvWithBaselines(validation_data)
        info = rollout_eval(ppo, val_env)
        return float(info.get("total_return", 0.0))

    return objective

## Load data
Same CSV layout as `train_portfolio.ipynb`.

In [4]:
stock_data = load_stock_data(DEFAULT_TICKERS, DATA_DIR)
training_data, validation_data, test_data = split_data(stock_data)
print(f"Train days: {len(next(iter(training_data.values())))}")
print(f"Val days:   {len(next(iter(validation_data.values())))}")

Train days: 1491
Val days:   253


## Run study

In [5]:
N_TRIALS = 20
TRAIN_ITERATIONS = 80
STEPS_PER_ITER = 2048
STUDY_NAME = "ppo_portfolio"
STORAGE = None  # e.g. "sqlite:///optuna_ppo.db"
SEED = 0

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="maximize",
    load_if_exists=bool(STORAGE),
)
objective = build_objective(
    training_data,
    validation_data,
    train_iterations=TRAIN_ITERATIONS,
    steps_per_iter=STEPS_PER_ITER,
    torch_seed=SEED,
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

[I 2026-04-04 22:08:08,008] A new study created in memory with name: ppo_portfolio


  0%|          | 0/20 [00:00<?, ?it/s]

Iteration 10/80 | Mean Reward: 0.09 | Policy Loss: -0.0201 | Value Loss: 0.0114 | Entropy: 29.9902
Iteration 20/80 | Mean Reward: 0.09 | Policy Loss: -0.0211 | Value Loss: 0.0050 | Entropy: 30.0960
Iteration 30/80 | Mean Reward: 0.08 | Policy Loss: -0.0250 | Value Loss: 0.0062 | Entropy: 30.1825
Iteration 40/80 | Mean Reward: 0.09 | Policy Loss: -0.0213 | Value Loss: 0.0032 | Entropy: 30.2908
Iteration 50/80 | Mean Reward: 0.09 | Policy Loss: -0.0165 | Value Loss: 0.0022 | Entropy: 30.2892
Iteration 60/80 | Mean Reward: 0.10 | Policy Loss: -0.0241 | Value Loss: 0.0093 | Entropy: 30.2259
Iteration 70/80 | Mean Reward: 0.09 | Policy Loss: -0.0195 | Value Loss: 0.0098 | Entropy: 30.2073
Iteration 80/80 | Mean Reward: 0.09 | Policy Loss: -0.0226 | Value Loss: 0.0144 | Entropy: 30.3491
[I 2026-04-04 22:35:16,933] Trial 0 finished with value: 34.42625673539646 and parameters: {'lr': 0.00045378889711869205, 'gamma': 0.9618757938060448, 'gae_lambda': 0.9208878686517381, 'clip_range': 0.2083662

In [5]:
t = study.best_trial
print("Best trial:")
print(f"  value (val total_return): {t.value}")
print(f"  params: {t.params}")

Best trial:
  value (val total_return): 48.720047223453065
  params: {'lr': 4.339007941494558e-05, 'gamma': 0.9691497927891632, 'gae_lambda': 0.9398215725667363, 'clip_range': 0.25174373908981257, 'n_epochs': 11, 'batch_size': 256, 'ent_coef': 0.003162917429834776, 'vf_coef': 0.6071689668113389}


In [ ]:
# plot the training curve
